# Pratique ML — Classification (scoring de risque de défaut)

**Objectif entretien Data Scientist / ML Engineer.**

Cible : `defaulted` (1 = l'entreprise fait défaut dans les 12 mois, 0 sinon).

## Mode d'emploi
- Remplis chaque cellule marquée `# TODO`. L'ordre suit un vrai pipeline d'entretien.
- Le **résultat attendu** est indiqué en commentaire au-dessus de chaque TODO.
- Corrigé complet : [`solutions/classification_xgboost.py`](solutions/classification_xgboost.py). Ne le regarde **qu'après** avoir tenté.
- Chronomètre-toi : ~40 min cible.

## Plan
0. Setup (fourni) · 1. Exploration · 2. Split stratifié · 3. Préprocessing · 4. Baseline LogisticRegression · 5. RandomForest + XGBoost · 6. Déséquilibre · 7. Métriques · 8. Cross-validation · 9. Seuil métier

## 0. Setup (fourni — exécute simplement)
Charge les données ; régénère le CSV automatiquement s'il manque.

In [ ]:
from pathlib import Path
import subprocess, sys
import numpy as np
import pandas as pd

RANDOM_STATE = 42
TARGET = "defaulted"
CATEGORICAL_FEATURES = ["country", "sector"]
NUMERIC_FEATURES = ["revenue", "debt_ratio", "days_late", "num_employees", "credit_score", "years_in_business"]
FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

DATA_PATH = Path("data/credit_risk.csv")
if not DATA_PATH.exists():
    subprocess.run([sys.executable, "generate_dataset.py"], check=True)
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## 1. Exploration
Comprends le déséquilibre des classes avant tout.

**Questions :** Quel est le taux de défaut ? Le problème est-il déséquilibré ? Y a-t-il des valeurs manquantes ?

In [ ]:
# TODO : afficher le taux de défaut (moyenne de la cible),
#        la distribution des classes (value_counts normalisé),
#        et le nombre de valeurs manquantes par colonne.
# Attendu : ~20-30% de défaut, 0 valeur manquante.


## 2. Split train / test stratifié
**Question :** pourquoi stratifier sur la cible quand les classes sont déséquilibrées ?

In [ ]:
from sklearn.model_selection import train_test_split

X, y = df[FEATURES], df[TARGET]

# TODO : créer X_train, X_test, y_train, y_test
#        test_size=0.2, random_state=RANDOM_STATE, stratify=y
# Attendu : le taux de défaut doit être ~identique entre train et test.

# print(f"Train {len(X_train)} | Test {len(X_test)}")
# print(f"Taux défaut train {y_train.mean():.2%} | test {y_test.mean():.2%}")


## 3. Préprocessing (ColumnTransformer)
One-hot sur les catégorielles, standardisation sur les numériques.

**Question :** pourquoi `handle_unknown="ignore"` pour le OneHotEncoder ?

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# TODO : construire un ColumnTransformer 'preprocessor' :
#   - ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES)
#   - ("num", StandardScaler(), NUMERIC_FEATURES)
# Attendu : preprocessor.fit_transform(X_train).shape[1] > len(FEATURES) (one-hot ajoute des colonnes).


## 4. Baseline — LogisticRegression
Toujours une baseline simple AVANT les modèles complexes.

**Question :** que représente une probabilité prédite de 0.9 pour un client ?

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score

# TODO : Pipeline [preprocessor -> LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)]
#        fit sur train, prédire sur test, afficher F1 et ROC-AUC.
# Attendu : ROC-AUC nettement > 0.5 (modèle informatif).


## 5. RandomForest + XGBoost
Compare des modèles non linéaires à la baseline.

**Question :** pourquoi un arbre/forêt n'a pas besoin de standardisation, contrairement à la régression logistique ?

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# TODO : entraîner RandomForestClassifier(class_weight='balanced') et XGBClassifier
#        dans des pipelines avec le preprocessor, comparer leur F1 / ROC-AUC à la baseline.
# Astuce XGBoost déséquilibre : scale_pos_weight = (y_train==0).sum() / (y_train==1).sum()
# Attendu : F1 >= celui de la baseline.


## 6. Gérer le déséquilibre
**Question :** différence entre `class_weight='balanced'`, `scale_pos_weight` et SMOTE ? Lequel rééchantillonne réellement les données ?

In [ ]:
# TODO : essayer SMOTE via imblearn.pipeline.Pipeline (imblearn) :
#   from imblearn.over_sampling import SMOTE
#   from imblearn.pipeline import Pipeline as ImbPipeline
#   ImbPipeline([("preprocessor", preprocessor), ("smote", SMOTE(random_state=RANDOM_STATE)), ("clf", ...)])
# Compare le recall avec / sans SMOTE.
# Attendu : le recall sur la classe 'défaut' augmente généralement.


## 7. Métriques détaillées
**Question :** dans la détection de défaut/fraude, pourquoi le recall est-il souvent plus critique que l'accuracy ?

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score

# TODO : pour ton meilleur modèle, afficher la matrice de confusion,
#        precision, recall, F1, ROC-AUC, et le classification_report.
# Identifie les faux négatifs (défauts manqués) = risque métier le plus coûteux.


## 8. Cross-validation stratifiée
**Question :** pourquoi une CV stratifiée donne une estimation plus robuste qu'un seul split ?

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# TODO : évaluer ton meilleur pipeline en CV 5 folds stratifiée (scoring='f1'),
#        afficher moyenne ± écart-type.
# Attendu : un écart-type faible = modèle stable.


## 9. Choix du seuil métier
Le seuil 0.5 n'est pas sacré. En assurance-crédit, manquer un défaut coûte cher.

**Question :** que se passe-t-il (precision vs recall) si on baisse le seuil de 0.5 à 0.3 ?

In [ ]:
# TODO : récupérer y_proba = model.predict_proba(X_test)[:, 1].
#        Balayer des seuils de 0.1 à 0.9 et tracer / afficher precision et recall pour chacun.
#        Choisir un seuil qui maximise le recall sous une contrainte de precision acceptable.
# Attendu : baisser le seuil augmente le recall mais baisse la precision.


---
**Auto-évaluation :** une fois terminé, compare ton raisonnement avec [`solutions/classification_xgboost.py`](solutions/classification_xgboost.py) (`python solutions/classification_xgboost.py` depuis `machine_learning/`).